# Transform sprints Data

1. Read bronze _sprints_ table
2. Keep only the columns required for analytics (Drop _url_ column).
3. Standardise column names using snake_case (_positionText_ -> _finish_position_text_ for example)
4. Rename columns to make them more meaningful (_date_ -> _race_date_ , _grid_ -> _grid_position_ for example)
5. Filter out rows where _season_ , _round_ , _constructor_id_ or _driver_id_ is null (BusinessKey validation)
6. Remove duplicate records
7. Transform values of column _race_name_ to Title Case
8. Write the transformed data to silver _sprints_ table.

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read bronze _sprints_ table

In [0]:
# Methode 1 that allows options since it's an API call:
# sprints_df = spark.read.table(bronze_table)

In [0]:
# Method 2 using a method with no additionnal options
sprints_df = spark.table(bronze_table)

In [0]:
display(sprints_df)

**2. Keep only the columns required for analytics (Drop _url_ column).**

In [0]:
sprints_dropped_df = sprints_df.drop("url")

In [0]:
display(sprints_dropped_df)

**3. Standardise column names using snake_case &
4. Rename columns to make them more meaningful**

In [0]:
# ANCIENNE VERSION!!
# sprints_renamed_df = (
#     sprints_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat", "latitude")
#     .withColumnRenamed("long","longitude")                       
# )

In [0]:
sprints_renamed_df = (
    sprints_dropped_df
    .withColumnsRenamed({
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName":"race_name",
        "date":"race_date",
        "grid":"grid_position",
        "laps":"completed_laps",
        "number":"car_number",
        "position":"final_position",
        "positionText":"final_position_text"
    })                     
)

In [0]:
display(sprints_renamed_df)

**5. Filter out rows where _season_ , _round_ , _constructor_id_ or _driver_id_ is null (BusinessKey validation)**

In [0]:
sprints_valid_df = (
    sprints_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)

In [0]:
display(sprints_valid_df)

**6. Remove duplicate records**

In [0]:
# sprints_distinct_df = sprints_valid_df.distinct()

In [0]:
sprints_distinct_df = sprints_valid_df.dropDuplicates(["season", "round", "constructor_id" , "driver_id"])

In [0]:
display(sprints_distinct_df)

**7. Transform values of column race_name to Title Case**

In [0]:
sprints_final_df = (
    sprints_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

**7. Write the transformed data to silver sprints table.**

In [0]:
(
    sprints_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))